In [ ]:
# === IMPORT LIBRARIES ===

# Web scraping
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service

# Data processing
import pandas as pd
import numpy as np

# Date and time
from datetime import datetime
import time
import re

# Visualization (optional)
try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    print("Matplotlib không được cài đặt. Sẽ bỏ qua việc vẽ biểu đồ.")
    HAS_MATPLOTLIB = False

print("Đã import tất cả libraries cần thiết")

✅ Đã import tất cả libraries cần thiết


In [ ]:
# === CONFIGURATION ===

# Danh sách các đồng coin cần crawl
COIN_LIST = [
    'btc', 'eth', 'bnb', 'sol', 'xrp', 'doge', 'ada', 'avax', 'trx', 'link',
    'dot', 'matic', 'ltc', 'bch', 'near', 'uni', 'atom', 'xlm', 'algo', 'vet'
]

# Cấu hình crawling
CRAWL_CONFIG = {
    'delay_between_coins': 2,  # giây
    'max_retries': 3,
    'timeout': 10,
    'batch_size': 5
}

# XPath selectors
XPATH_SELECTORS = {
    'price': '//span[@class="text-xl font-semibold text-text-primary"]',
    'market_cap': '//div[contains(text(), "Market Cap")]/following-sibling::div',
    'volume': '//div[contains(text(), "24h Volume")]/following-sibling::div'
}

print(f"Cấu hình hoàn tất")
print(f"- Số coins cần crawl: {len(COIN_LIST)}")
print(f"- Delay giữa các coins: {CRAWL_CONFIG['delay_between_coins']} giây")

✅ Cấu hình hoàn tất
📋 Số lượng coins sẽ crawl: 10
🏷️ Danh sách coins: bitcoin, ethereum, binance, solana, ripple, cardano, avalanche, polkadot, chainlink, tron


In [ ]:
# === UTILITY FUNCTIONS ===

def extract_date_from_string(date_string):
    """
    Trích xuất và format ngày từ chuỗi text
    
    Args:
        date_string (str): Chuỗi chứa thông tin ngày tháng
        
    Returns:
        str: Ngày được format theo YYYY-MM-DD, hoặc None nếu không parse được
    """
    try:
        date_pattern = r'(\w+ \d+, \d{4})'
        match = re.search(date_pattern, date_string)
        if match:
            parsed_date = datetime.strptime(match.group(1), '%B %d, %Y')
            return parsed_date.strftime('%Y-%m-%d')
        return None
    except Exception as e:
        print(f"Lỗi khi parse ngày '{date_string}': {e}")
        return None

def convert_market_cap_to_number(value):
    """
    Chuyển đổi giá trị market cap từ string sang số
    
    Args:
        value (str): Giá trị market cap (ví dụ: "$1.2B", "$500M")
        
    Returns:
        float: Giá trị số, hoặc None nếu không convert được
    """
    if pd.isna(value) or not value:
        return None
        
    # Loại bỏ ký tự không cần thiết
    clean_value = value.replace('$', '').replace(',', '').strip()
    
    # Xác định multiplier
    multiplier = 1
    if clean_value.endswith('B'):
        multiplier = 1_000_000_000
        clean_value = clean_value[:-1]
    elif clean_value.endswith('M'):
        multiplier = 1_000_000
        clean_value = clean_value[:-1]
    elif clean_value.endswith('K'):
        multiplier = 1_000
        clean_value = clean_value[:-1]
    
    try:
        return float(clean_value) * multiplier
    except ValueError:
        print(f"Không thể convert market cap: {value}")
        return None

def get_coin_data(driver, coin_name):
    """
    Thu thập dữ liệu của một đồng coin từ Bitget
    
    Args:
        driver: Selenium WebDriver instance
        coin_name (str): Tên của coin cần crawl
        
    Returns:
        dict: Dictionary chứa thông tin coin, hoặc error info nếu có lỗi
    """
    url = f"{CRAWL_CONFIG['base_url']}{coin_name}"
    
    try:
        print(f"Đang crawl {coin_name.upper()}...")
        driver.get(url)
        time.sleep(CRAWL_CONFIG['wait_time'])
        
        # Lấy các elements
        labels = driver.find_elements(By.XPATH, XPATH_SELECTORS['labels'])
        values = driver.find_elements(By.XPATH, XPATH_SELECTORS['values'])
        price_element = driver.find_element(By.XPATH, XPATH_SELECTORS['price'])
        date_element = driver.find_element(By.XPATH, XPATH_SELECTORS['date'])
        
        # Extract text và clean data
        keys = [el.text.replace(':', '').strip() for el in labels]
        vals = [el.text.replace('\n', '').strip() for el in values]
        
        # Tạo dictionary từ keys và values
        limit = min(len(keys), len(vals))
        data = {keys[i]: vals[i] for i in range(limit)}
        
        # Thêm thông tin cơ bản
        data.update({
            'coin': coin_name,
            'price': price_element.text.strip(),
            'date': date_element.text.strip(),
            'crawl_timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        })
        
        print(f"Crawl {coin_name.upper()} thành công - Giá: {data['price']}")
        return data
        
    except Exception as e:
        error_msg = f"Lỗi khi crawl {coin_name}: {str(e)}"
        print(f"{error_msg}")
        return {
            'coin': coin_name,
            'error': str(e),
            'crawl_timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }

print("Đã định nghĩa các utility functions")

✅ Đã định nghĩa các utility functions


In [ ]:
# === SINGLE CRAWL FUNCTIONS ===

def process_dataframe(df):
    """
    Xử lý và clean DataFrame
    
    Args:
        df (pandas.DataFrame): DataFrame raw
        
    Returns:
        pandas.DataFrame: DataFrame đã được clean
    """
    if df.empty:
        return df
    
    # Thêm parsed date
    if 'date' in df.columns:
        df['parsed_date'] = df['date'].apply(extract_date_from_string)
    
    # Sắp xếp lại columns theo thứ tự ưu tiên
    priority_columns = ['coin', 'price', 'date', 'parsed_date', 'crawl_timestamp']
    other_columns = [col for col in df.columns 
                    if col not in priority_columns and col != 'error']
    
    # Tạo column order
    column_order = priority_columns + other_columns
    if 'error' in df.columns:
        column_order.append('error')
    
    # Reorder columns
    existing_columns = [col for col in column_order if col in df.columns]
    df = df[existing_columns]
    
    # Set index
    if 'coin' in df.columns:
        df.set_index('coin', inplace=True)
    
    return df

def crawl_all_coins_once(coin_list=None):
    """
    Thu thập dữ liệu tất cả coins một lần
    
    Args:
        coin_list (list): Danh sách coins cần crawl. Nếu None thì dùng COIN_LIST
        
    Returns:
        pandas.DataFrame: DataFrame chứa dữ liệu đã được clean
    """
    if coin_list is None:
        coin_list = COIN_LIST
    
    print(f"Bắt đầu crawl {len(coin_list)} coins...")
    print("=" * 50)
    
    # Khởi tạo driver
    driver = webdriver.Chrome()
    
    try:
        # Thu thập dữ liệu
        all_data = []
        for coin in coin_list:
            coin_data = get_coin_data(driver, coin)
            all_data.append(coin_data)
        
        # Tạo DataFrame
        df = pd.DataFrame(all_data)
        
        # Clean và process data
        df = process_dataframe(df)
        
        print("=" * 50)
        print(f"Hoàn tất crawl {len(coin_list)} coins")
        print(f"Tổng số bản ghi: {len(df)}")
        
        return df
        
    except Exception as e:
        print(f"Lỗi trong quá trình crawl: {e}")
        return pd.DataFrame()
        
    finally:
        driver.quit()
        print("Đã đóng trình duyệt")

print("Đã định nghĩa functions cho single crawl")

✅ Đã định nghĩa functions cho single crawl


In [ ]:
# === THỰC HIỆN CRAWL DỮ LIỆU MỘT LẦN ===

print("Bắt đầu crawl dữ liệu tất cả coins...")
df_single_crawl = crawl_all_coins_once()

# Hiển thị kết quả
print("\nKẾT QUẢ CRAWL:")
if not df_single_crawl.empty:
    print(f"Kích thước DataFrame: {df_single_crawl.shape}")
    print(f"Các cột có sẵn: {list(df_single_crawl.columns)}")
    print("\n" + "="*50)
    df_single_crawl
else:
    print("Không có dữ liệu được crawl")

🎯 Bắt đầu crawl dữ liệu tất cả coins...
🚀 Bắt đầu crawl 10 coins...
🔍 Đang crawl BITCOIN...
🔍 Đang crawl BITCOIN...
✅ Crawl BITCOIN thành công - Giá: $105,397.31
🔍 Đang crawl ETHEREUM...
✅ Crawl BITCOIN thành công - Giá: $105,397.31
🔍 Đang crawl ETHEREUM...
✅ Crawl ETHEREUM thành công - Giá: ₫68,511,922.02
🔍 Đang crawl BINANCE...
✅ Crawl ETHEREUM thành công - Giá: ₫68,511,922.02
🔍 Đang crawl BINANCE...
✅ Crawl BINANCE thành công - Giá: ₫17,506,975.2
🔍 Đang crawl SOLANA...
✅ Crawl BINANCE thành công - Giá: ₫17,506,975.2
🔍 Đang crawl SOLANA...
✅ Crawl SOLANA thành công - Giá: ₫4,079,876.21
🔍 Đang crawl RIPPLE...
✅ Crawl SOLANA thành công - Giá: ₫4,079,876.21
🔍 Đang crawl RIPPLE...
✅ Crawl RIPPLE thành công - Giá: ₫58,283.05
🔍 Đang crawl CARDANO...
✅ Crawl RIPPLE thành công - Giá: ₫58,283.05
🔍 Đang crawl CARDANO...
✅ Crawl CARDANO thành công - Giá: ₫18,186.5
🔍 Đang crawl AVALANCHE...
✅ Crawl CARDANO thành công - Giá: ₫18,186.5
🔍 Đang crawl AVALANCHE...
✅ Crawl AVALANCHE thành công - Giá: 

In [ ]:
# === PHÂN TÍCH MARKET CAP ===

def analyze_market_cap(df):
    """
    Phân tích và trực quan hóa dữ liệu Market Cap
    
    Args:
        df (pandas.DataFrame): DataFrame chứa dữ liệu coins
    """
    if df.empty:
        print("DataFrame rỗng, không thể phân tích Market Cap")
        return
    
    if 'Market Cap' not in df.columns:
        print("Không tìm thấy cột 'Market Cap' trong dữ liệu")
        print(f"Các cột có sẵn: {list(df.columns)}")
        return
    
    print("Đang phân tích Market Cap...")
    
    # Convert Market Cap sang số
    df_analysis = df.copy()
    df_analysis['market_cap_value'] = df['Market Cap'].apply(convert_market_cap_to_number)
    
    # Loại bỏ các giá trị None
    valid_data = df_analysis.dropna(subset=['market_cap_value'])
    
    if valid_data.empty:
        print("Không có dữ liệu Market Cap hợp lệ để phân tích")
        return
    
    # Sắp xếp theo Market Cap
    market_cap_sorted = valid_data.sort_values('market_cap_value', ascending=False)
    
    print(f"Đã phân tích {len(market_cap_sorted)} coins có Market Cap hợp lệ")
    
    # Hiển thị top coins
    print("\nTOP COINS THEO MARKET CAP:")
    print("-" * 60)
    for idx, (coin, row) in enumerate(market_cap_sorted.head().iterrows(), 1):
        market_cap_display = row['Market Cap']
        price = row.get('price', 'N/A')
        print(f"{idx:2d}. {coin.upper():12s} | {market_cap_display:>12s} | Price: {price}")
    
    # Vẽ biểu đồ nếu có matplotlib
    if HAS_MATPLOTLIB:
        try:
            plt.figure(figsize=(14, 8))
            
            # Tạo bar chart
            coins = [coin.upper() for coin in market_cap_sorted.index[:10]]  # Top 10
            values = market_cap_sorted['market_cap_value'].head(10).values
            
            bars = plt.bar(coins, values, color='steelblue', alpha=0.7)
            
            # Customize chart
            plt.title('Market Cap của Top 10 Cryptocurrency', fontsize=16, fontweight='bold')
            plt.xlabel('Cryptocurrency', fontsize=12)
            plt.ylabel('Market Cap (USD)', fontsize=12)
            plt.xticks(rotation=45, ha='right')
            
            # Format y-axis
            plt.ticklabel_format(style='plain', axis='y')
            
            # Thêm giá trị lên đầu mỗi bar
            for bar, value in zip(bars, values):
                height = bar.get_height()
                if value >= 1e9:
                    label = f'${value/1e9:.1f}B'
                elif value >= 1e6:
                    label = f'${value/1e6:.1f}M'
                else:
                    label = f'${value/1e3:.1f}K'
                
                plt.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                        label, ha='center', va='bottom', fontsize=10)
            
            plt.tight_layout()
            plt.grid(axis='y', alpha=0.3)
            plt.show()
            
            print("\nĐã vẽ biểu đồ Market Cap thành công")
            
        except Exception as e:
            print(f"Không thể vẽ biểu đồ: {e}")
    
    # Hiển thị bảng nếu không có matplotlib hoặc lỗi vẽ biểu đồ
    if not HAS_MATPLOTLIB:
        print("\nTHÔNG TIN MARKET CAP (Dạng bảng):")
        display_columns = ['Market Cap', 'market_cap_value', 'price']
        available_columns = [col for col in display_columns if col in market_cap_sorted.columns]
        print(market_cap_sorted[available_columns].head(10))
    
    return market_cap_sorted

# Thực hiện phân tích Market Cap
if 'df_single_crawl' in locals() and not df_single_crawl.empty:
    market_cap_analysis = analyze_market_cap(df_single_crawl)
else:
    print("Cần crawl dữ liệu trước khi phân tích Market Cap")

⚠️ Không tìm thấy cột 'Market Cap' trong dữ liệu
📋 Các cột có sẵn: ['price', 'date', 'parsed_date', 'crawl_timestamp', 'Market cap', 'Fully diluted market cap', 'Volume (24h)', '24h volume / market cap', '24h high', '24h low', 'All-time high', 'All-time low', 'Circulating supply', 'Total supply', 'Circulation rate', 'Max supply', 'Price in BTC', 'Price in ETH', 'Price at BTC market cap', 'Price at ETH market cap', 'Contracts', 'Links']


In [ ]:
# === CONTINUOUS CRAWLING FUNCTION ===

def crawl_continuously(coin_list=None, interval_minutes=5, max_updates=10, save_to_file=False):
    """
    Thu thập dữ liệu cryptocurrency theo định kỳ
    
    Args:
        coin_list (list): Danh sách coins cần crawl
        interval_minutes (int): Thời gian giữa các lần crawl (phút)
        max_updates (int): Số lần crawl tối đa
        save_to_file (bool): Có lưu dữ liệu ra file không
        
    Returns:
        pandas.DataFrame: DataFrame chứa tất cả dữ liệu đã crawl
    """
    if coin_list is None:
        coin_list = COIN_LIST
    
    print(f"BẮT ĐẦU CONTINUOUS CRAWLING")
    print(f"Coins: {', '.join([c.upper() for c in coin_list])}")
    print(f"Interval: {interval_minutes} phút")
    print(f"Số lần crawl: {max_updates}")
    print("=" * 60)
    
    # Khởi tạo driver và DataFrame
    driver = webdriver.Chrome()
    master_df = pd.DataFrame()
    
    try:
        for update_num in range(1, max_updates + 1):
            current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            print(f"\nUPDATE #{update_num}/{max_updates} - {current_time}")
            print("-" * 40)
            
            # Thu thập dữ liệu cho tất cả coins
            batch_data = []
            for coin in coin_list:
                coin_data = get_coin_data(driver, coin)
                coin_data['update_number'] = update_num
                coin_data['batch_timestamp'] = current_time
                batch_data.append(coin_data)
            
            # Tạo DataFrame cho batch hiện tại
            batch_df = pd.DataFrame(batch_data)
            batch_df['parsed_date'] = batch_df['date'].apply(extract_date_from_string)
            
            # Thêm vào master DataFrame
            if master_df.empty:
                master_df = batch_df.copy()
            else:
                master_df = pd.concat([master_df, batch_df], ignore_index=True)
            
            # Hiển thị thông tin batch
            print(f"Crawl batch #{update_num} hoàn tất")
            print(f"Tổng bản ghi: {len(master_df)}")
            
            # Hiển thị giá hiện tại
            print("Giá hiện tại:")
            for data in batch_data:
                if 'error' not in data:
                    print(f"  {data['coin'].upper():12s}: {data['price']}")
                else:
                    print(f"  {data['coin'].upper():12s}: Error")
            
            # Lưu file nếu được yêu cầu
            if save_to_file:
                filename = f"crypto_data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
                master_df.to_csv(filename, index=False)
                print(f"Đã lưu dữ liệu vào {filename}")
            
            # Chờ đến lần crawl tiếp theo
            if update_num < max_updates:
                print(f"Đợi {interval_minutes} phút cho lần crawl tiếp theo...")
                time.sleep(interval_minutes * 60)
    
    except KeyboardInterrupt:
        print("\nNgười dùng đã dừng quá trình crawling")
    except Exception as e:
        print(f"\nLỗi trong quá trình crawling: {e}")
    finally:
        driver.quit()
        print("\nĐã đóng trình duyệt")
    
    print(f"\nHOÀN TẤT CONTINUOUS CRAWLING")
    print(f"Tổng số bản ghi thu thập: {len(master_df)}")
    
    return master_df

print("Đã định nghĩa function continuous crawling")

✅ Đã định nghĩa function continuous crawling
